# 📘 Notebook 05: Learning Step by Step: SARSA and Q-learning

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module B: Learning with a Table · Notebook 1 of 3 in this module · (5 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain **temporal-difference (TD) learning**: updating an estimate from the next estimate, without waiting for the end of the episode
- Identify the **TD target** and the **TD error** in an update rule
- Implement **SARSA** and **Q-learning** and state the one line in which they differ
- Explain **on-policy** and **off-policy** learning with the Cliff Walking experiment
- Handle `terminated` and `truncated` correctly when computing a target
- Check that Q-learning reaches the optimum that planning computes

> **Prerequisites:** Notebooks 01 to 04, in particular the Bellman equations (Notebook 03) and Q-tables with epsilon-greedy exploration (Notebook 04).
>
> 🔭 **Why this notebook matters later:** Q-learning is the most important algorithm of this course. Notebook 07 uses it to teach a snake, and Notebook 09 replaces its table with a neural network and calls the result DQN. The update rule stays the same throughout.

> 📦 **Libraries used in this notebook.** **Gymnasium** for the Cliff Walking, Frozen Lake and Taxi environments, **imageio** for the videos. The cells of this notebook take well under a minute in total.

In [ ]:
%pip install -q "gymnasium[toy-text]>=1.3" imageio imageio-ffmpeg

In [ ]:
import base64
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
from IPython.display import HTML, display


def show_video(frames, fps=6, width=560):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))

## 1. Do not wait for the end

### Intuition first
You leave work and expect to be home in 30 minutes. Five minutes later you are stuck behind an accident and now expect the rest of the trip to take 40 minutes. You do not need to arrive home to know that your first estimate was too low: 5 minutes spent plus 40 expected is 45, not 30. You can correct the estimate **now**.

Monte Carlo learning (Notebook 04) would wait until you open your front door. **Temporal-difference learning** corrects each estimate as soon as the next one is available.

### Formal definition
After one step from state $S$ to state $S'$ with reward $R$, the TD update for a state value is:

$$V(S) \leftarrow V(S) + \alpha\,\big[\,\underbrace{R + \gamma\,V(S')}_{\text{TD target}} - V(S)\,\big]$$

- The **TD target** $R + \gamma V(S')$ is a better-informed estimate of the value of $S$: one real reward, plus the current guess for the rest.
- The bracket is the **TD error**: how much the old estimate disagrees with the target.
- $\alpha$ (alpha) is the **step size** or **learning rate**.

It is the pattern of Notebook 01 again: new = old + step × (target − old). And the target is the right-hand side of a **Bellman equation**, with one sampled step in place of the sum over all possibilities.

Using an estimate to improve an estimate is called **bootstrapping**. It sounds circular, but every update mixes in one real reward, and the real information spreads through the table from there.

### Why it matters

| | Dynamic programming | Monte Carlo | Temporal difference |
|---|---|---|---|
| Needs the model | yes | no | **no** |
| Learns before the episode ends | (plays no episodes) | no | **yes** |
| Target | expected reward + value of next states | the complete return | one reward + value of the next state |

TD learning keeps the advantage of each of the other two.

## 2. SARSA

### Formal definition
For control we need action values (Notebook 04, Section 4). Apply the TD idea to $Q$. The agent is in state $S$, takes action $A$, receives reward $R$, arrives in $S'$ and **chooses its next action** $A'$. Then:

$$Q(S, A) \leftarrow Q(S, A) + \alpha\,\big[\,R + \gamma\,Q(S', A') - Q(S, A)\,\big]$$

The update uses the five items $S, A, R, S', A'$, which give the algorithm its name.

## 3. Q-learning

### Formal definition
**Q-learning** (Chris Watkins, 1989) changes one thing. The target does not use the action the agent will actually take next. It uses the **best** action available in $S'$ according to the current table:

$$Q(S, A) \leftarrow Q(S, A) + \alpha\,\big[\,R + \gamma\,\max_{a} Q(S', a) - Q(S, A)\,\big]$$

Compare with Notebook 03. SARSA's target is a sample of the Bellman **expectation** equation, for the policy the agent is following. Q-learning's target is a sample of the Bellman **optimality** equation.

Both algorithms fit in one function. Look for the two lines that set `target`.

In [ ]:
def choose_action(Q, state, epsilon, rng):
    """Epsilon-greedy. Ties between equally good actions are broken at random."""
    if rng.random() < epsilon:
        return int(rng.integers(Q.shape[1]))
    best = np.flatnonzero(Q[state] == Q[state].max())
    return int(rng.choice(best))


def train(env, method, episodes, alpha=0.5, gamma=1.0, epsilon=0.1, decay=False, max_steps=1000, seed=0):
    """Tabular SARSA or Q-learning on an environment with numbered states. Returns Q and the reward of each episode."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.observation_space.n, env.action_space.n))
    episode_rewards = []
    for episode in range(episodes):
        eps = epsilon * (1 - episode / episodes) if decay else epsilon      # optionally fade exploration out
        state, info = env.reset(seed=seed * 100000 + episode)
        action = choose_action(Q, state, eps, rng)
        total = 0.0
        for t in range(max_steps):
            next_state, reward, terminated, truncated, info = env.step(action)
            next_action = choose_action(Q, next_state, eps, rng)
            future = 0.0 if terminated else 1.0                 # nothing follows a terminal state
            if method == "sarsa":
                target = reward + gamma * future * Q[next_state, next_action]     # the action actually chosen
            else:
                target = reward + gamma * future * Q[next_state].max()            # the best action
            Q[state, action] += alpha * (target - Q[state, action])
            state, action = next_state, next_action
            total += reward
            if terminated or truncated:
                break
        episode_rewards.append(total)
    return Q, np.array(episode_rewards)

> 🧠 **A detail that matters.** Notebook 02 promised that the difference between `terminated` and `truncated` would become important. Here it is. When the game is really over (`terminated`), no reward can follow, so the target is the reward alone. When the episode was merely cut off by a time limit (`truncated`), the state still had a future, and the target keeps its $\gamma\,Q(S', \cdot)$ part. Treating a time limit as a real ending teaches the agent that running out of time is worth zero, and silently distorts the values.

## 4. The cliff

To see what the single different line does, we use a classic experiment. In **Cliff Walking** the agent starts at the bottom left of a 4 × 12 grid and must reach the bottom right. Every step costs −1. The bottom edge between start and goal is a cliff: stepping off it costs −100 and sends the agent back to the start.

In [ ]:
cliff = gym.make("CliffWalking-v1", render_mode="rgb_array")
state, info = cliff.reset(seed=0)
print("states:", cliff.observation_space.n, "  actions:", cliff.action_space.n, "(0 up, 1 right, 2 down, 3 left)")

plt.figure(figsize=(9, 3))
plt.imshow(cliff.render())
plt.axis("off")
plt.show()

The shortest route runs along the edge of the cliff: 13 steps, a return of −13. A safer route goes up and across the top.

Both agents train for 500 episodes with a fixed epsilon of 0.1, so one move in ten is random for the whole experiment. We repeat the training with 10 different seeds and average the curves, for the reason given in Notebook 01.

In [ ]:
def average_curves(env, method, runs=10, episodes=500):
    curves = [train(env, method, episodes, seed=seed)[1] for seed in range(runs)]
    return np.mean(curves, axis=0)


def smooth(values, window=20):
    return np.convolve(values, np.ones(window) / window, mode="valid")


curve_sarsa = average_curves(cliff, "sarsa")
curve_q = average_curves(cliff, "q_learning")

plt.figure(figsize=(7, 4))
plt.plot(smooth(curve_sarsa), label="SARSA")
plt.plot(smooth(curve_q), label="Q-learning")
plt.ylim(-120, 0)
plt.title("Cliff Walking: reward per episode while learning")
plt.xlabel("episode"); plt.ylabel("total reward (average of 10 runs, smoothed)")
plt.grid(True); plt.legend()
plt.show()

print(f"average reward over the last 100 episodes:  SARSA {curve_sarsa[-100:].mean():.1f},  Q-learning {curve_q[-100:].mean():.1f}")

While learning, **SARSA collects clearly more reward than Q-learning**. That looks like a defeat for the algorithm we called the most important of the course. Before judging, look at what each agent has actually learned. We switch exploration off and follow the greedy policy of each table.

In [ ]:
ARROWS = ["↑", "→", "↓", "←"]


def show_policy(Q, title):
    print(title)
    for row in range(4):
        cells = []
        for col in range(12):
            state = row * 12 + col
            if row == 3 and 0 < col < 11:
                cells.append("▒")                        # the cliff
            elif state == 47:
                cells.append("G")
            else:
                cells.append(ARROWS[int(Q[state].argmax())])
        print("   " + " ".join(cells))


def greedy_episode(env, Q, max_steps=200, seed=0):
    state, info = env.reset(seed=seed)
    frames, total = [env.render()], 0.0
    for t in range(max_steps):
        state, reward, terminated, truncated, info = env.step(int(Q[state].argmax()))
        total += reward
        frames.append(env.render())
        if terminated or truncated:
            break
    return total, frames


Q_sarsa, rewards = train(cliff, "sarsa", 500, seed=0)
Q_q, rewards = train(cliff, "q_learning", 500, seed=0)

show_policy(Q_sarsa, "SARSA:")
return_sarsa, frames_sarsa = greedy_episode(cliff, Q_sarsa)
print(f"   return when followed greedily: {return_sarsa:.0f}\n")

show_policy(Q_q, "Q-learning:")
return_q, frames_q = greedy_episode(cliff, Q_q)
print(f"   return when followed greedily: {return_q:.0f}")

Follow the arrows from the bottom-left corner. Arrows in squares that the agent hardly ever visits are unreliable, so look only at the route.

- **Q-learning** has learned the **optimal** route, straight along the edge of the cliff, with a return of −13.
- **SARSA** has learned a **safer, longer** route that keeps away from the edge.

In [ ]:
print("SARSA, greedy:")
show_video(frames_sarsa)

In [ ]:
print("Q-learning, greedy:")
show_video(frames_q)

## 5. On-policy and off-policy

### Intuition first
Why did the agent with the better route earn less while learning? Because during learning both agents were still taking a random step one time in ten, and a random step next to a cliff is expensive.

- **SARSA** learns the value of the policy **it is actually following**, random steps included. Walking along the edge is genuinely bad for someone who stumbles every tenth step, and SARSA's values say so. It learns the best behaviour *for a clumsy walker*.
- **Q-learning** learns the value of the **greedy** policy, whatever it is doing at the moment. Its `max` assumes that from the next state on it will act perfectly. It learns the best behaviour *for a sure-footed walker*, while still being clumsy itself.

### Formal definition
- The **behaviour policy** is the one that chooses the actions and generates the experience.
- The **target policy** is the one whose values are being learned.
- An algorithm is **on-policy** when the two are the same (SARSA: both epsilon-greedy), and **off-policy** when they differ (Q-learning: behaviour is epsilon-greedy, target is greedy).

### Why it matters
Off-policy learning means an agent can learn about the best behaviour **from experience produced by a different behaviour**: by an exploring version of itself, by an older version of itself, even by a human. Notebook 09 depends on this. DQN stores old experience in a memory and learns from it again and again, long after the policy that produced it has changed. Only an off-policy method is allowed to do that.

> ⚠️ **Common pitfalls**
>
> - **Judging an agent by its reward during training.** That reward includes the cost of exploring. Evaluate the greedy policy separately, as we did.
> - **Bootstrapping from a terminal state.** After `terminated`, the target is the reward alone.
> - **A step size that is too large in a noisy environment.** With `alpha = 0.5` every update half-replaces the old estimate. That is fine on the deterministic cliff and too jumpy on the slippery lake below.
> - **Believing the arrows in unvisited states.** A Q-table is only trustworthy where the agent has been.

## 6. Does Q-learning find the true optimum?

In Notebook 03 we computed the best possible policy for a slippery Frozen Lake by planning with the model. Q-learning has no model. Does it arrive at the same quality of play, from experience alone?

The lake is slippery (the character slides sideways two times out of three), so the rewards are noisy. We use a smaller step size and let exploration fade from 1 to 0 over the training.

In [ ]:
lake = gym.make("FrozenLake-v1", is_slippery=True)


def success_rate(env, Q, episodes=2000):
    wins = 0.0
    for episode in range(episodes):
        state, info = env.reset(seed=10000 + episode)
        terminated = truncated = False
        while not (terminated or truncated):
            state, reward, terminated, truncated, info = env.step(int(Q[state].argmax()))
        wins += reward
    return 100 * wins / episodes


# Q-learning: experience only
Q_lake, rewards = train(lake, "q_learning", 20000, alpha=0.1, gamma=0.99, epsilon=1.0, decay=True)

# Value iteration, as in Notebook 03: uses the model lake.unwrapped.P
P = lake.unwrapped.P
V = np.zeros(16)
for sweep in range(2000):
    V = np.array([max(sum(p * (r + 0.99 * (0.0 if done else V[s2])) for p, s2, r, done in P[s][a])
                      for a in range(4)) for s in range(16)])
Q_planned = np.array([[sum(p * (r + 0.99 * (0.0 if done else V[s2])) for p, s2, r, done in P[s][a])
                       for a in range(4)] for s in range(16)])

print(f"Q-learning (no model):    reaches the present in {success_rate(lake, Q_lake):.0f}% of episodes")
print(f"value iteration (model):  reaches the present in {success_rate(lake, Q_planned):.0f}% of episodes")

The two agree to within a percentage point or two. On this lake even perfect play fails about one time in four, because of the sliding and the 100-step time limit, and the agent that only ever saw its own experience plays as well as the one that was given the rules.

Under mild conditions (every state-action pair keeps being tried, and the step size is reduced suitably) Q-learning is **proven** to converge to the optimal action values $q_*$. We have now seen it happen.

---
## ✏️ Exercises

The exercises use `train`, `choose_action`, `show_policy`, `greedy_episode`, `smooth` and `show_video` from above.

### Exercise 1 (One update by hand)
A Q-table has `Q[s] = [2.0, 5.0]` and `Q[s'] = [1.0, 4.0]`. The agent is in `s`, takes action 0, receives a reward of −1 and lands in `s'`, where its epsilon-greedy policy happens to choose action 0. With `alpha = 0.5` and `gamma = 0.9`, compute the new value of `Q[s][0]` (a) with SARSA and (b) with Q-learning. Do it on paper, then check in Python.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
alpha, gamma, reward = 0.5, 0.9, -1.0
Q_s, Q_next = [2.0, 5.0], [1.0, 4.0]

sarsa = Q_s[0] + alpha * (reward + gamma * Q_next[0] - Q_s[0])           # the action chosen: 0
q_learning = Q_s[0] + alpha * (reward + gamma * max(Q_next) - Q_s[0])    # the best action: 1
print(f"SARSA:      {sarsa:.2f}")
print(f"Q-learning: {q_learning:.2f}")
```

*SARSA: target −1 + 0.9 × 1 = −0.1, new value 2 + 0.5 × (−0.1 − 2) = 0.95. Q-learning: target −1 + 0.9 × 4 = 2.6, new value 2.3. The exploratory choice in `s'` lowers SARSA's estimate and leaves Q-learning's untouched.*
</details>

### Exercise 2 (Letting exploration fade)
Train SARSA and Q-learning on the cliff for 500 episodes with `epsilon=0.5, decay=True`, so that exploration fades to zero. Print each greedy return and show both policies. Once SARSA no longer stumbles, the edge of the cliff is safe for it. Does it switch to the short route?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for method in ("sarsa", "q_learning"):
    Q, rewards = train(cliff, method, 500, epsilon=0.5, decay=True, seed=0)
    total, frames = greedy_episode(cliff, Q)
    show_policy(Q, f"{method}:")
    print(f"   greedy return: {total:.0f},  average reward over the last 50 training episodes: {rewards[-50:].mean():.1f}\n")
```

*It does not. SARSA's values for the squares along the edge were learned while it was still stumbling, and they are pessimistic. As epsilon fades, the agent becomes greedy with respect to those values, stops visiting the edge, and so never finds out that the edge has become safe. In theory SARSA reaches the optimal policy if exploration fades slowly enough. In practice that can take a very long time. An on-policy learner can only correct what it still visits.*
</details>

### Exercise 3 (The step size)
Section 6 trained on the slippery lake with `alpha=0.1`. Repeat that training (20,000 episodes, `gamma=0.99, epsilon=1.0, decay=True`) with `alpha=0.9`, for three different seeds each, and compare with `alpha=0.1`. For every run print the success rate and the largest difference between the learned table and `Q_planned`. What goes wrong with the large step?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for alpha in (0.1, 0.9):
    for seed in range(3):
        Q, rewards = train(lake, "q_learning", 20000, alpha=alpha, gamma=0.99, epsilon=1.0, decay=True, seed=seed)
        print(f"alpha = {alpha}, seed {seed}: success rate {success_rate(lake, Q, episodes=500):3.0f}%,  "
              f"largest error in the table {np.abs(Q - Q_planned).max():.2f}")
```

*On the slippery lake the same action in the same state leads to different places. With a step of 0.9 each estimate is almost entirely replaced by the latest outcome, so the table never settles and its errors stay large. Where two actions are close in value, the noise decides between them, and the quality of the final policy becomes a matter of luck: compare the three seeds. A small step averages over many outcomes.*
</details>

### Exercise 4 (A taxi driver)
`Taxi-v4` has 500 states and 6 actions: a taxi must drive to a passenger, pick them up, drive to the destination and drop them off. Each step costs −1, a successful drop-off gives +20, and an illegal pick-up or drop-off costs −10. Train Q-learning for 3,000 episodes (`alpha=0.5, gamma=0.99, epsilon=0.1, max_steps=200`), plot the smoothed reward per episode, and show a video of the greedy policy.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
taxi = gym.make("Taxi-v4", render_mode="rgb_array")
Q_taxi, rewards = train(taxi, "q_learning", 3000, alpha=0.5, gamma=0.99, epsilon=0.1, max_steps=200)

plt.figure(figsize=(7, 4))
plt.plot(smooth(rewards, 50))
plt.title("Taxi: reward per episode while learning")
plt.xlabel("episode"); plt.ylabel("total reward (smoothed)"); plt.grid(True)
plt.show()

total, frames = greedy_episode(taxi, Q_taxi, seed=3)
print(f"greedy episode: return {total:.0f} in {len(frames) - 1} steps")
show_video(frames, fps=3, width=420)
```

*Nothing in `train` was changed. The same function that learned a walk along a cliff learns a four-stage delivery task, because all it needs is numbered states, numbered actions and rewards.*
</details>

### Exercise 5 (No exploration at all)
Train Q-learning on the cliff for 500 episodes with `epsilon=0.0`. Does it still find the optimal route? Now do the same on the **non-slippery** Frozen Lake (`gym.make("FrozenLake-v1", is_slippery=False)`, `gamma=0.99`) for 2,000 episodes and measure the success rate. In each case, explain what makes the agent try new things although it never explores on purpose.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
Q, rewards = train(cliff, "q_learning", 500, epsilon=0.0, seed=0)
print(f"cliff, epsilon 0: greedy return {greedy_episode(cliff, Q)[0]:.0f}")

dry_lake = gym.make("FrozenLake-v1", is_slippery=False)
Q, rewards = train(dry_lake, "q_learning", 2000, gamma=0.99, epsilon=0.0, seed=0)
print(f"dry lake, epsilon 0: success rate {success_rate(dry_lake, Q, episodes=100):.0f}%, "
      f"episodes that ever reached the present during training: {int(rewards.sum())}")
```

*Both agents succeed, for different reasons. On the cliff every step gives −1, so the value of whatever the agent has just tried drops below the untried actions, which are still at 0. The greedy agent is pushed towards everything it has not done yet: optimism again, as in Notebook 01. On the lake all rewards are 0 until the present is reached, so nothing distinguishes one action from another. What saves the agent is a detail of `choose_action`: ties are broken **at random**, so it wanders like a random walker until it stumbles on the present, and from then on the reward spreads back through the table. Had ties always gone to the first action, the agent would have pushed "left" against the wall for ever. When rewards are rare, exploration has to come from somewhere.*
</details>

### Exercise 6 (Expected SARSA (a little harder))
SARSA's target uses the one action that happened to be chosen next, which adds noise. **Expected SARSA** replaces it by the **average** over the next actions, weighted by the probabilities of the epsilon-greedy policy: each action has probability `epsilon / n_actions`, and the greedy action has an additional `1 - epsilon`. Write `train_expected_sarsa`, run it on the cliff with `alpha=0.5` and `epsilon=0.1` for 10 seeds, and compare its average reward over the last 100 episodes with SARSA and Q-learning.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
def train_expected_sarsa(env, episodes, alpha=0.5, gamma=1.0, epsilon=0.1, max_steps=1000, seed=0):
    rng = np.random.default_rng(seed)
    n_actions = env.action_space.n
    Q = np.zeros((env.observation_space.n, n_actions))
    episode_rewards = []
    for episode in range(episodes):
        state, info = env.reset(seed=seed * 100000 + episode)
        total = 0.0
        for t in range(max_steps):
            action = choose_action(Q, state, epsilon, rng)
            next_state, reward, terminated, truncated, info = env.step(action)
            probabilities = np.full(n_actions, epsilon / n_actions)
            probabilities[Q[next_state].argmax()] += 1 - epsilon
            expected = probabilities @ Q[next_state]                 # average over the next actions
            target = reward + gamma * (0.0 if terminated else expected)
            Q[state, action] += alpha * (target - Q[state, action])
            state = next_state
            total += reward
            if terminated or truncated:
                break
        episode_rewards.append(total)
    return Q, np.array(episode_rewards)

curve_expected = np.mean([train_expected_sarsa(cliff, 500, seed=seed)[1] for seed in range(10)], axis=0)
print(f"SARSA:          {curve_sarsa[-100:].mean():7.1f}")
print(f"Q-learning:     {curve_q[-100:].mean():7.1f}")
print(f"Expected SARSA: {curve_expected[-100:].mean():7.1f}")
```

*Expected SARSA evaluates the same epsilon-greedy policy as SARSA, so it also learns to respect the cliff, but its target does not depend on the luck of one sampled action. With `epsilon = 0` its target becomes the maximum, and it turns into Q-learning.*
</details>

## 🔑 Key takeaways

- **Temporal-difference** learning updates an estimate towards **reward + discounted next estimate**, after every step, with no model.
- The **TD error** is target minus old estimate. Using estimates as targets is called **bootstrapping**.
- **SARSA** uses the next action actually chosen; **Q-learning** uses the best next action. One line of code separates them.
- SARSA is **on-policy**: it learns the value of the exploring policy it follows, and plays safe. Q-learning is **off-policy**: it learns the value of the greedy policy while behaving differently.
- After `terminated` the target is the reward alone. After `truncated` the future still counts.
- With enough exploration, Q-learning reaches the same optimum that planning computes from the model.

---
**Next:** *Notebook 06: Building Snake*, where we write a real game as an environment for these algorithms.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*